In [1]:
import pandas as pd
from scipy import stats
import numpy as np
from pandas.api.types import CategoricalDtype
import re
from helpers import get_renamed_blocks, get_renamed_blocks3

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
#STATES = ['in Betrieb', 'Gesetzlich an Stilllegung gehindert', 'Netzreserve',  'Sicherheitsbereitschaft', 'Sonderfall', 'vorläufig stillgelegt', 'stillgelegt', 'Kohlestromvermarktungsverbot']
#STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'Strommarktrückkehr', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt KVBG', 'stillgelegt']
STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'bnBm', 'KVBG','stillgelegt', 'vorläufig stillgelegt']
ACTIVE_STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'bnBm', 'vorläufig stillgelegt']
ENERGIES = ["Kernenergie", "Braunkohle", "Steinkohle", "Erdgas", "Mineralölprodukte", "Abfall", "Biomasse", ""]

In [6]:
bm = pd.read_csv("../basic/inspire_prtr_mapper.csv", engine="python")
plr = pd.read_excel("../data/Kraftwerksliste.xlsx", sheet_name=1, skiprows=10)
pl0 = pd.read_excel("../data/Kraftwerksliste.xlsx", skiprows=11)
prtr = pd.read_excel("../data/2023-12-08_PRTR-Deutschland_Freisetzungen.xlsx")

In [7]:
bm['PRTR_Kennnummer'] = bm['PRTR_Kennnummer'].apply(lambda x: str(x).replace('/', '_'))
bm['InspireID_Betrieb'] = bm['InspireID_Betrieb'].apply(lambda x: str(x).replace('/', '_'))

In [8]:
#bm.loc[bm.plantid.str.contains("_")]

In [9]:
pl0_bak = pl0.copy()
plr_bak = plr.copy()

In [10]:
#list(pl0_bak)
#list(plr_bak)

In [11]:
cols = [0,12,15,16,17,20]
plt = get_renamed_blocks(pl0)
plq = plt.drop(plt.columns[cols],axis=1)
cols2 = [0, 10,11,17,20,21]
plr0a = get_renamed_blocks3(plr)
plr0b = plr0a.drop(plr0a.columns[cols2],axis=1)

In [12]:
#list(plq)
#list(plr0b)

In [13]:
plq0a = plq[~plq['blockid'].isin(["SEE9-Dummy-nicht_EEG", "SEE9-Dummy-EEG"])] # remove non-conventional facilities
plq0b = plq0a.sort_values(['power'], ascending=False)

In [14]:
plq0a.insert(10, "endop", np.nan)
plr0b.insert(2, "company", np.nan)

In [15]:
plq0a.shape

(1842, 19)

In [16]:
plq0a.shape

(1842, 19)

In [17]:
plq0a.shape

(1842, 19)

In [18]:
plr0b.shape

(273, 17)

In [19]:
pl_combined = pd.concat([plq0a, plr0b])
pl_combined['energysource'] = pl_combined['energysource'].apply(lambda x: str(x).replace('Wärme', 'Erdgas'))

In [20]:
pl_combined

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
0,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023-04-05 00:00:00,2023.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,21.5050,NaN,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN,NaN
1,SEE946736241334,Palm Power GmbH & Co. KG,HKW Aalen,73432,Aalen,Palm Allee,1,Baden-Württemberg,2021-11-03 00:00:00,2021.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,78.3020,NaN,Volleinspeisung,Netze BW GmbH (SNB948311994307),Hochspannung; Hochspannung,NaN,NaN
2,SEE930596800480,Lausitz Energie Kraftwerke AG,Gasturbinenkraftwerk Ahrensfelde GT A,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990-12-22 00:00:00,1990.0,NaN,Kapazitätsreserve aufgrund von § 13e EnWG,"Erdgas, Erdölgas",Erdgas,37.1040,NaN,Volleinspeisung,50Hertz Transmission GmbH (SNB982046657236),Hochspannung,NaN,NaN
3,SEE929797382345,Lausitz Energie Kraftwerke AG,Gasturbinenkraftwerk Ahrensfelde GT B,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990-12-10 00:00:00,1990.0,NaN,Kapazitätsreserve aufgrund von § 13e EnWG,"Erdgas, Erdölgas",Erdgas,37.1040,NaN,Volleinspeisung,50Hertz Transmission GmbH (SNB982046657236),Hochspannung,NaN,NaN
4,SEE988046628214,Lausitz Energie Kraftwerke AG,Gasturbinenkraftwerk Ahrensfelde GT C,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1991-03-26 00:00:00,1991.0,NaN,Kapazitätsreserve aufgrund von § 13e EnWG,"Erdgas, Erdölgas",Erdgas,37.1040,NaN,Volleinspeisung,50Hertz Transmission GmbH (SNB982046657236),Hochspannung,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,SEE994506958166,NaN,PSW Hohenwarte 1 PSS B,7338.0,Hohenwarte,Ortsstraße,NaN,Thüringen,1959-08-19 00:00:00,NaN,2024-07-04 00:00:00,NaN,NaN,Pumpspeicher,29.9000,NaN,Volleinspeisung,NaN,NaN,Nein,30.000
269,SEE926418118239,NaN,GTS4,49479.0,Ibbenbüren,Groner Allee,76,Nordrhein-Westfalen,2013-06-07 00:00:00,NaN,2024-07-31 00:00:00,NaN,"Erdgas, Erdölgas",Erdgas,1.8223,NaN,Teileinspeisung (einschließlich Eigenverbrauch),SWTE Netz GmbH & Co. KG (SNB950006175489),NaN,Ja,1.853
270,SEE937157344278,NaN,Kraftwerk Walheim Block 1,74399.0,Walheim,Mühlstraße,1,Baden-Württemberg,1964-09-10 00:00:00,NaN,2024-07-31 00:00:00,NaN,Steinkohlen,Steinkohle,96.0000,NaN,Volleinspeisung,Netze BW GmbH (SNB948311994307),NaN,Nein,107.000
271,SEE981220191160,NaN,HKW Oberkirch,77704.0,Oberkirch,Hauptstraße,2,Baden-Württemberg,1987-05-26 00:00:00,NaN,2024-08-30 00:00:00,NaN,Steinkohlen,Steinkohle,18.5000,NaN,Teileinspeisung (einschließlich Eigenverbrauch),NaN,NaN,Ja,20.000


In [21]:
pl_combined.loc[pl_combined.blockid == "BNA0711"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
8,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963,NaN,2012,NaN,NaN,Braunkohle,125.0,NaN,NaN,Amprion GmbH,NaN,Nein,NaN


In [22]:
pl_combined.shape

(2115, 21)

In [23]:
pl0 = pl_combined.copy()

In [24]:
pl_debug = pl0.copy()
pl_debug['initialop'] = pd.to_datetime(pl_debug['initialop'], errors='coerce')
pl_debug2 = pl_debug.copy()
pl1 = pl0.loc[pl0["energysource"].isin(ENERGIES)]
pl_debug = pl0.copy()
pl_debug['initialop'] = pd.to_datetime(pl_debug['initialop'], errors='coerce')
pl_debug2 = pl_debug.copy()

In [25]:
pl1 = pl0.loc[pl0["energysource"].isin(ENERGIES)]

In [26]:
pl0.loc[pl0["blockid"] == "BNA0645"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower


In [27]:
pl_debug2.groupby("state").count()

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
state,,,,,,,,,,,,,,,,,,,,
In Betrieb,1766,1766,1766,1766,1766,1675,1535,1766,1766,1766,0,1229,1766,1766,64,1734,1766,1766,0,0
Kapazitätsreserve aufgrund von § 13e EnWG,16,16,16,16,16,16,15,16,16,16,0,16,16,16,0,16,16,16,0,0
Netzreserve aufgrund von KVBG,4,4,4,4,4,4,4,4,4,4,0,4,4,4,0,4,4,4,0,0
Netzreserve aufgrund § 13b EnWG,23,23,23,23,23,23,22,23,23,23,0,21,23,23,0,23,23,23,0,0
besonderes netztechnisches Betriebsmittel,14,14,14,14,14,14,14,14,14,14,0,13,14,14,0,14,14,14,0,0
in Betrieb,2,2,2,2,2,2,2,2,2,2,0,2,2,2,0,2,2,2,0,0
vorläufig stillgelegt nach § 13b EnWG,8,8,8,8,8,8,7,8,8,8,0,8,8,8,0,8,8,8,0,0
vorläufig stillgelegt ohne § 13b EnWG,9,9,9,9,9,9,9,9,9,9,0,9,9,9,0,9,9,9,0,0


In [28]:
#pl_debug2.groupby("initialop").count()

In [29]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

/tmp/ipykernel_1714311/293089823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl1['state'] = pl1['state'].fillna("stillgelegt")


In [30]:
pl1.loc[pl1.blockid == "SEE975094128629"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
266,SEE975094128629,NaN,HKW Mitte Kessel 12,38114.0,Braunschweig,Reiherstraße,3,Niedersachsen,1971-07-15 00:00:00,NaN,2024-06-26 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,20.0,NaN,Volleinspeisung,NaN,NaN,Ja,22.4


In [31]:
#pl_debug2['initialop'] = pl_debug.initialop.dt.year

In [32]:
#pl_debug['initialop'] = pl_debug['initialop'].to_datetime()

In [33]:
pl_debug2.groupby("state").count()

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
state,,,,,,,,,,,,,,,,,,,,
In Betrieb,1766,1766,1766,1766,1766,1675,1535,1766,1766,1766,0,1229,1766,1766,64,1734,1766,1766,0,0
Kapazitätsreserve aufgrund von § 13e EnWG,16,16,16,16,16,16,15,16,16,16,0,16,16,16,0,16,16,16,0,0
Netzreserve aufgrund von KVBG,4,4,4,4,4,4,4,4,4,4,0,4,4,4,0,4,4,4,0,0
Netzreserve aufgrund § 13b EnWG,23,23,23,23,23,23,22,23,23,23,0,21,23,23,0,23,23,23,0,0
besonderes netztechnisches Betriebsmittel,14,14,14,14,14,14,14,14,14,14,0,13,14,14,0,14,14,14,0,0
in Betrieb,2,2,2,2,2,2,2,2,2,2,0,2,2,2,0,2,2,2,0,0
vorläufig stillgelegt nach § 13b EnWG,8,8,8,8,8,8,7,8,8,8,0,8,8,8,0,8,8,8,0,0
vorläufig stillgelegt ohne § 13b EnWG,9,9,9,9,9,9,9,9,9,9,0,9,9,9,0,9,9,9,0,0


In [34]:
#pl_debug2.groupby("initialop").count()

In [35]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

/tmp/ipykernel_1714311/293089823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl1['state'] = pl1['state'].fillna("stillgelegt")


In [36]:
#pl0

In [37]:
#bm

In [38]:
#pl0.groupby("Energieträger").count()

In [39]:
#list(plt)

In [40]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

/tmp/ipykernel_1714311/293089823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl1['state'] = pl1['state'].fillna("stillgelegt")


In [41]:
pl1.loc[pl1.blockid == "BNA0221c"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
108,BNA0221c,NaN,Gasblock,40221.0,Düsseldorf,Auf der Lausward,75,Nordrhein-Westfalen,1977-03-28 00:00:00,NaN,2019,stillgelegt,NaN,Erdgas,293.0,NaN,NaN,Netzgesellschaft Düsseldorf mbH,NaN,Ja,NaN


In [42]:
plt.loc[plt.blockid == "BNA0711"]

,Unnamed: 0,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,state,e1,e2,energysource,chp,border,grosspower,power,gerpower,tech,fullsupply,TSO,voltagelevel


In [43]:
plt = pl1.dropna(subset=["blockid"])

In [44]:
plt = plt.astype({"energysource": 'category'})

In [45]:
plt.loc[plt.blockid == "BNA0711"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
8,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963,NaN,2012,stillgelegt,NaN,Braunkohle,125.0,NaN,NaN,Amprion GmbH,NaN,Nein,NaN


In [46]:
pl2 = pl1.copy()

In [47]:
def to_year(x):
    dt = pd.to_datetime(x)
    return dt.year

In [48]:
def to_year2(x):
    if pd.isna(x):
        return x
    elif isinstance(x, float):
        return x
    elif isinstance(x, int):
        return x
    else:
        ts = pd.Timestamp(ts_input=x)
        #print(ts)
        return int(ts.year)

In [49]:
#blocks.loc[blocks.Energieträger == "Kernenergie"]

In [50]:
#pl2

In [51]:
'''
pl2.loc[pl2.blockid == 'BNA0861a', 'initialop'] = 2012
pl2.loc[pl2.blockid == 'BNA1334', 'initialop'] = 2002
pl2.loc[pl2.blockid == 'BNA1141', 'initialop'] = 1970
pl2.loc[pl2.blockid == 'BNA0418', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1499', 'initialop'] = 1951
pl2.loc[pl2.blockid == 'BNA1502', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1500', 'initialop'] = 1990
pl2.loc[pl2.blockid == 'BNA1498', 'initialop'] = 1953
pl2.loc[pl2.blockid == 'BNA1260', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1056', 'initialop'] = 2006
pl2.loc[pl2.blockid == 'BNA1114', 'initialop'] = 2012



pl2.loc[pl2.blockid == 'BNA0413b', 'initialop'] = 2014 # Westfalen D

pl2.loc[pl2.blockid == 'BNA0355', 'initialop'] = 1981 # Grafenrheinfeld
'''

#pl2['Nettoleistung'] = pl2['Nettoleistung'].str.replace('\n','')
#pl2['Nettoleistung'] = pl2[pd.to_numeric(pl2["Nettoleistung"])]
#pl2['Nettoleistung'] = pl2['Nettoleistung'].str.replace(';','.')

"\npl2.loc[pl2.blockid == 'BNA0861a', 'initialop'] = 2012\npl2.loc[pl2.blockid == 'BNA1334', 'initialop'] = 2002\npl2.loc[pl2.blockid == 'BNA1141', 'initialop'] = 1970\npl2.loc[pl2.blockid == 'BNA0418', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1499', 'initialop'] = 1951\npl2.loc[pl2.blockid == 'BNA1502', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1500', 'initialop'] = 1990\npl2.loc[pl2.blockid == 'BNA1498', 'initialop'] = 1953\npl2.loc[pl2.blockid == 'BNA1260', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1056', 'initialop'] = 2006\npl2.loc[pl2.blockid == 'BNA1114', 'initialop'] = 2012\n\n\n\npl2.loc[pl2.blockid == 'BNA0413b', 'initialop'] = 2014 # Westfalen D\n\npl2.loc[pl2.blockid == 'BNA0355', 'initialop'] = 1981 # Grafenrheinfeld\n"

In [52]:
#pl2[] = pl2[pd.to_numeric(pl2["Nettoleistung"])]
pl2['initialop'] = pl2['initialop'].apply(to_year2)
#pl2['endop'] = pl2['endop'].apply(to_year2)
pl2['power'] = pl2['power'].apply(pd.to_numeric)

In [53]:
pl2.loc[pl2.blockid == "BNA0711"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
8,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963.0,NaN,2012,stillgelegt,NaN,Braunkohle,125.0,NaN,NaN,Amprion GmbH,NaN,Nein,NaN


In [54]:
#bm[pl2.duplicated(['BlockID'], keep=False)].sort_values("BlockID", ascending=False)

In [55]:
#pl2[pl2.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

In [56]:
#cols = [10,11,12,14,17,18,19]
#pl3 = pl2.drop(pl2.columns[cols],axis=1)

In [57]:
#pl3.groupby('Unternehmen').max()

In [58]:
#pl3

In [59]:
def fix_company(company):
    #print(company)
    company_dict = {"RWE": "RWE AG", "Vattenfall": "Vattenfall GmbH", "Uniper": "Uniper SE", "EnBW": "EnBW AG", "Steag": "Steag GmbH", "Nordzucker": "Nordzucker AG", "Lausitz Energie": "LEAG"}
    for key, value in company_dict.items():
        if key in str(company):
            return value
    return company

In [60]:
def extract_still(x):
    match = re.findall("[0-9]{4}",x)
    return match[0] if match else np.nan

In [61]:
def fix_kwk(x):
    return "Nein" if x == "nein" else "Ja" if x == "ja" else x

In [62]:
pl4 = pl2.copy()

In [63]:
sq = 'Endgültig Stillgelegt 2013 (mit StA)'

In [64]:
list(pl2.groupby("state").count().reset_index()['state'])

['In Betrieb',
 'Kapazitätsreserve aufgrund von § 13e EnWG',
 'Netzreserve aufgrund von KVBG',
 'Netzreserve aufgrund § 13b EnWG',
 'besonderes netztechnisches Betriebsmittel',
 'in Betrieb',
 'stillgelegt',
 'vorläufig stillgelegt nach § 13b EnWG',
 'vorläufig stillgelegt ohne § 13b EnWG']

In [65]:
z = "endgültig stillgelegt 2022 (ohne § 13b EnWG oder KVBG)"

In [66]:
def extract_state(state):
    if state == "In Betrieb":
        return "in Betrieb"
    elif "Endgültig Stillgelegt" in state:
        return "stillgelegt"
    elif "Vorläufig Stillgelegt" in state:
        return "vorläufig stillgelegt"
    elif "an Stilllegung gehindert" in state:
        return "Gesetzlich an Stilllegung gehindert"
    else:
        return state

In [67]:
def extract_state2(state):
    
    states = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'Strommarktrückkehr', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt KVBG', 'stillgelegt']
    matches = ['In Betrieb', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve aufgrund § 13b EnWG', 'befristete Strommarktrückkehr', 'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'endgültig stillgelegt 20[0-9]{2} \(nach KVBG\)', r'[E|e]ndgültig [S|s]tillgelegt 20[0-9]{2} \([ohne]|[mit].']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [68]:
def extract_state3(state):
    
    states = ['in Betrieb', 'bnBm', 'KVBG', 'Kapazitätsreserve', 'Netzreserve', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt']
    matches = ['In Betrieb', 'besonderes netztechnisches Betriebsmittel', r'Netzreserve aufgrund von KVBG', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve',  r'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'stillgelegt']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [69]:
extract_state3(sq)

'Endgültig Stillgelegt 2013 (mit StA)'

In [70]:
#pl4

In [71]:
pl4['company'] = pl4['company'].apply(lambda x: fix_company(x))
#pl4['endop'] = pl4['state'].apply(lambda x: extract_still(x))
pl4['state'] = pl4['state'].apply(lambda x: extract_state3(x))
pl4['chp'] = pl4['chp'].apply(lambda x: fix_kwk(x))

In [72]:
list(pl4.groupby("state").count().reset_index()['state'])

['KVBG',
 'Kapazitätsreserve',
 'Netzreserve',
 'bnBm',
 'in Betrieb',
 'stillgelegt',
 'vorläufig stillgelegt']

In [73]:
atest = pl4.groupby("state").count()

In [74]:
#atest

In [75]:
newdf = pl4.copy()

In [76]:
#newdf.dtypes

In [77]:
newdf["state"] = newdf["state"].astype('category')
newdf["state"] = newdf["state"].cat.set_categories(STATES, ordered=True)

In [78]:
newdf["chp"] = newdf["chp"].astype('category')
newdf["chp"] = newdf["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)

In [79]:
pl3 = newdf

In [80]:
atest = newdf.groupby(["power"]).count()
atest.sort_values("initialop", inplace=True, ascending=False)

In [81]:
#print(atest)

In [82]:
atest = pl3.groupby(["state"]).count()
# atest

/tmp/ipykernel_1714311/4069360528.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  atest = pl3.groupby(["state"]).count()


In [83]:
# pl3.rename(columns={ pl3.columns[10]: "Energieträger"}, inplace=True)

In [84]:
#bm[bm.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

In [85]:
#bm2 = bm.drop_duplicates()

In [86]:
#pl3

In [87]:
#bm

In [88]:
pl4 = bm.merge(pl3, how="right", left_on="sseid", right_on="blockid")

In [89]:
pl5 = pl4[pd.notnull(pl4['blockid'])]
#pl5 = pl4
pl5a = pl5[pd.notnull(pl5['power'])]
#pl6 =  pl5a.copy() #[pd.notnull(pl5a['plantid'])]
pl6 = pl5a.rename(columns={"InspireID_Betrieb": "plantid"})

In [90]:
plnewtest = pl6.dropna(subset="plantid").sort_values("plantid")

In [91]:
#plnewtest

In [92]:
#pl6.dtypes

In [93]:
pl6.loc[pl6.blockid == "BNA0711"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
1297,NW300-0326774,06-05-300-0326774,BNA0711,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963.0,NaN,2012,stillgelegt,NaN,Braunkohle,125.0,NaN,NaN,Amprion GmbH,NaN,Nein,NaN


In [94]:
pl6['initialop'] = pl6['initialop'].apply(pd.to_numeric)
pl6['endop'] = pl6['endop'].apply(lambda x: to_year2(x))

In [95]:
#pl6.loc[pl6.blockid == "BNA0711"].dtypes

In [96]:
pl6.loc[pl6.blockid == "SEE999977738880"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
1419,SD664-02,664-02,SEE999977738880,SEE999977738880,NaN,IKW Deuben,6682.0,Teuchern,Industriestraße,1,Sachsen-Anhalt,1936.0,NaN,2021.0,stillgelegt,Rohbraunkohlen,Braunkohle,67.0,NaN,Teileinspeisung (einschließlich Eigenverbrauch),NaN,NaN,Ja,78.0


In [97]:
blocks = pl6.copy()
stammdaten = pl6.copy()

In [98]:
plants = blocks.copy()
#plants = plants.dropna(subset=["plantid", "initialop"]) #TODO: remove initialop from here

In [99]:
plants["energysource"] = plants["energysource"].astype('category')
plants["energysource"] = plants["energysource"].cat.set_categories(ENERGIES, ordered=True)
plants["state"] = plants["state"].astype('category')
#plants["state"] = plants["state"].cat.set_categories(STATES, ordered=True)
plants["chp"] = plants["chp"].astype('category')
plants["chp"] = plants["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)
plants["federalstate"] = plants["federalstate"].astype('category')
plants["federalstate"] = plants["federalstate"].cat.set_categories(['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'])

In [100]:
pl1 = pl0.loc[pl0["energysource"].isin(["Kernenergie", "Erdgas", "Steinkohle", "Braunkohle", "Steinkohle"])]

In [101]:
p_grp = plants.groupby("plantid")

In [102]:
#plants.groupby("Energip_subgrper").max()

In [103]:
plants.groupby("state").count()

/tmp/ipykernel_1714311/326027710.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  plants.groupby("state").count()


,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
state,,,,,,,,,,,,,,,,,,,,,,,
in Betrieb,823,823,823,1215,1215,1215,1215,1215,1149,1126,1215,1215,1215,0,1194,1215,1215,0,1205,1215,1215,0,0
Kapazitätsreserve,12,12,12,16,16,16,16,16,16,15,16,16,16,0,16,16,16,0,16,16,16,0,0
Netzreserve,23,23,23,23,23,23,23,23,23,22,23,23,23,0,21,23,23,0,23,23,23,0,0
bnBm,14,14,14,14,14,14,14,14,14,14,14,14,14,0,13,14,14,0,14,14,14,0,0
KVBG,4,4,4,4,4,4,4,4,4,4,4,4,4,0,4,4,4,0,4,4,4,0,0
stillgelegt,207,207,207,249,0,246,248,249,183,164,249,248,0,249,150,249,249,0,126,168,0,248,135
vorläufig stillgelegt,13,13,13,17,17,17,17,17,17,16,17,17,17,0,17,17,17,0,17,17,17,0,0


In [104]:
ACTIVE_STATES

['in Betrieb',
 'Kapazitätsreserve',
 'Netzreserve',
 'bnBm',
 'vorläufig stillgelegt']

In [105]:
#plants.loc[plants['state'] == 'in Betrieb']

In [106]:
#plants.loc[plants['state'].isin(ACTIVE_STATES)].sort_values('power', ascending=False)

In [107]:
p_subgrp = plants.loc[plants['state'].isin(ACTIVE_STATES)].groupby('plantid') # TODO: add 

In [108]:
plants_act = pd.DataFrame()
for plantid, group in p_subgrp:
    entry = {}
    entry["plantid"] = plantid
    entry["activepower"] = group["power"].sum()
    #plants_act = plants_act.append(entry, ignore_index=True)
    plants_act = pd.concat([plants_act, pd.DataFrame([entry])], ignore_index=True)

In [109]:
#plants_act

In [110]:
plants_a = pd.DataFrame()
for plantid, group in p_grp:
    #if not plantid == "06-05-300-9046797":
    #    continue
    entry = {}
    #print(entry)
    entry["plantid"] = plantid
    # entry["BlockID"] = group
    try:
        entry["plantname"] = group["plantname"].value_counts().index[0]
    except IndexError:
        entry["plantname"] = np.nan
    entry["federalstate"] = group["federalstate"].value_counts().index[0]
    entry["energysource"] = group["energysource"].min()
    try:
        entry["chp"] = group["chp"].min()
    except TypeError:
        #print("aneror")
        #print(plantid)
        #print(group['KWK'].count())
        entry["chp"] = ""
    entry["latestexpanded"] = group["initialop"].max()
    entry["initialop"] = group["initialop"].min()
    entry["totalpower"] = group["power"].sum()
    entry["state"] = group["state"].min()
    entry["blockcount"] = group["blockid"].count()
    try:
        entry["company"] = group["company"].value_counts().index[0]
    except IndexError:
        ;
    #print(entry)
    #break
    plants_a = pd.concat([plants_a, pd.DataFrame([entry])], ignore_index=True)

In [111]:
int_list = ['blockcount', 'initialop', 'latestexpanded']
for column in int_list:
    plants_a[column] = plants_a[column].astype(int)
#plant['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)

In [112]:
#plants_a.dtypes

In [113]:
plants_a["energysource"] = plants_a["energysource"].astype('category')
plants_a["energysource"] = plants_a["energysource"].cat.set_categories(ENERGIES, ordered=True)
plants_a["state"] = plants_a["state"].astype('category')
plants_a["state"] = plants_a["state"].cat.set_categories(STATES, ordered=True)
plants_a["chp"] = plants_a["chp"].astype('category')
plants_a["chp"] = plants_a["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)
plants_a["federalstate"] = plants_a["federalstate"].astype('category')
plants_a["federalstate"] = plants_a["federalstate"].cat.set_categories(['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'])

In [114]:
plants_a.loc[plants_a.plantid == "666-999"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [115]:
plants_a.loc[plants_a.plantid == "06-00176010435"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [116]:
plants_a.loc[plants_a.plantid == "06-05-900-0865327"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [117]:
#plants_a = plants.drop_duplicates(subset="KraftwerkID")

In [118]:
blocks.loc[blocks.sseid == "SEE930982693153"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
504,NW300-9046030,06-05-300-9046030,SEE930982693153,SEE930982693153,Knapsack Power GmbH & Co. KG,Knapsack I - Gasturbine GT11,50354,Hürth,Industriestraße,300,Nordrhein-Westfalen,2007.0,2007.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,308.0,NaN,Volleinspeisung,Amprion GmbH (SNB976890256486),Höchstspannung,NaN,NaN


In [119]:
column_titles = ['plantid', 'plantname', 'federalstate','energysource', 'chp', 'latestexpanded', 'initialop', 'totalpower', 'state', 'blockcount', 'company']
plants_b = plants_a.reindex(columns=column_titles)

In [120]:
blocks[blocks.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower
785,NaN,NaN,NaN,SEE988996320985,Caterpillar Energy Solutions,P31,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,1994.0,1994.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,1.84,NaN,Teileinspeisung (einschließlich Eigenverbrauch),MVV Netze GmbH (SNB985472799266),Mittelspannung,NaN,NaN
786,NaN,NaN,NaN,SEE988996320985,Caterpillar Energy Solutions,P31,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,1994.0,1994.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,1.84,NaN,Teileinspeisung (einschließlich Eigenverbrauch),MVV Netze GmbH (SNB985472799266),Mittelspannung,NaN,NaN


In [121]:
plants_b.sort_values("initialop", ascending=False, inplace=True)

In [122]:
plants_final = plants_b.loc[:]

In [123]:
#plants_final.dtypes

In [124]:
# plants_final

In [125]:
# stammdaten

In [126]:
#stammdaten

In [127]:
# cols = [0,2,3,9,10,11,12]
stammdaten = pl6.copy()
drop_list = ['plantid', 'energysource', 'initialop', 'chp', 'plantname', 'power', 'state', 'endop', 'company', 'sseid', 'TSO', 'voltagelevel']
stammdaten_dafuq = stammdaten.drop(drop_list, axis=1)
stammdaten = stammdaten_dafuq.copy()

In [128]:
stammdaten_dafuq.sort_values(['blockid'])

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,gerpower,fullsupply,grosspower
1294,06-08-4380932,BNA0011,79774.0,Albbruck,NaN,NaN,Baden-Württemberg,NaN,Steinkohle,NaN,NaN,NaN
1389,03-07-07244141350,BNA0012d,31061.0,Alfeld,Mühlenmarsch,1,Niedersachsen,NaN,NaN,NaN,NaN,NaN
1316,06-26200010633,BNA0059a,34225.0,Baunatal,NaN,NaN,Hessen,NaN,NaN,NaN,NaN,NaN
1392,06-11-01-1105607,BNA0075,12207.0,Berlin,Ostpreußendamm,61,Berlin,NaN,NaN,NaN,NaN,NaN
1375,06-11-01-1105607,BNA0076,12207.0,Berlin,Ostpreußendamm,61,Berlin,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
761,NaN,SEE999819576335,39126,Magdeburg,Kraftwerk-Privatweg,7,Sachsen-Anhalt,2006.0,"Abfall (Hausmüll, Siedl.abf.)",NaN,Volleinspeisung,NaN
254,06-05-100-0154540,SEE999839188105,40476,Düsseldorf,Rather Straße,51,Nordrhein-Westfalen,2012.0,"Erdgas, Erdölgas",NaN,Teileinspeisung (einschließlich Eigenverbrauch),NaN
1416,06-02-BERZ003800,SEE999848168525,21079.0,Hamburg,Moorburger Schanze,2,Hamburg,NaN,Steinkohlen,NaN,Volleinspeisung,860.0
969,NaN,SEE999966551940,14478,Potsdam,Zum Heizwerk,20,Brandenburg,1996.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN


In [129]:
stmp = pl6.copy()
stammdaten2 = stmp.merge(prtr, how="left", left_on="plantid", right_on="kennnummer")

In [130]:
#stammdaten2

In [131]:
DROP_ST = ['blockid', 'jahr', 'kennnummer','betriebsname','betriebsname_2','plz_x','ort','strasse','hausnr','bundesland','flusseinzugsgebiet','taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']
DROP_ST2 = ['jahr', 'kennnummer','betriebsname','betriebsname_2','plz','ort','strasse','hausnr','bundesland','flusseinzugsgebiet','taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']
DROP_ST3 = ['jahr', 'kennnummer', 'bundesland', 'flusseinzugsgebiet', "betriebsname", "betriebsname_2", 'taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']


In [132]:
#DROP_ST = ['jahr']

In [133]:
plants_d1 = plants_final.merge(prtr, how="left", left_on="plantid", right_on="kennnummer")

In [134]:
#plants_d1.dtypes

In [135]:
pld2 = plants_d1.drop_duplicates(['plantid'])

In [136]:
pld3 = pld2.drop(DROP_ST3, axis=1)
plants_final1 = pld3

In [137]:
#plants_act

In [138]:
plants_final2 = plants_final1.merge(plants_act, on="plantid", how='left')

In [139]:
plants_final3 = plants_final2.dropna(subset=["plantid"])

In [140]:
plants_final3.groupby("energysource").count()

/tmp/ipykernel_1714311/2887887728.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  plants_final3.groupby("energysource").count()


,plantid,plantname,federalstate,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
energysource,,,,,,,,,,,,,,,,,,,
Kernenergie,8,8,8,8,8,8,8,8,8,0,0,0,0,0,0,0,0,0,0
Braunkohle,29,29,29,15,29,29,29,29,29,24,0,0,0,0,0,0,0,0,24
Steinkohle,65,65,65,40,65,65,65,65,65,45,4,4,4,4,4,4,4,4,45
Erdgas,226,224,226,44,226,226,226,226,226,205,1,1,1,1,1,1,1,1,205
Mineralölprodukte,22,22,22,8,22,22,22,22,22,15,0,0,0,0,0,0,0,0,15
Abfall,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Biomasse,1,1,1,0,1,1,1,1,1,1,0,0,0,0,0,0,0,0,1
,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [141]:
plants_final2[pd.notnull(plants_final2['activepower'])].sort_values('activepower', ascending=False)

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
230,SN70015796,Boxberg Block N,Sachsen,Braunkohle,NaN,2012,1979,2470.000,in Betrieb,4,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2470.000
322,NW300-0326774,Niederaußem,Nordrhein-Westfalen,Braunkohle,Ja,2003,1963,3359.000,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2220.000
275,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,Ja,2012,1972,4211.000,in Betrieb,7,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2120.000
344,NW300-0877384,Weisweiler,Nordrhein-Westfalen,Braunkohle,Ja,2006,1955,2619.000,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2040.000
217,BB45025564,Kraftwerk Jänschwalde Block A,Brandenburg,Braunkohle,Ja,1989,1981,3000.000,in Betrieb,6,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,SN60018636,BHKW-G66-KWK-klein,Sachsen,Erdgas,NaN,2017,2017,3.736,in Betrieb,1,Volkswagen Sachsen GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.736
65,SD661-80,0050 - STW - BHKW - KWK,Nordrhein-Westfalen,Erdgas,Ja,2017,2007,12.877,in Betrieb,5,Stadtwerke Kempen GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.313
26,NW700-0104479,BHKW-G,Nordrhein-Westfalen,Erdgas,NaN,2016,2016,1.999,in Betrieb,1,Westag AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.999
24,NW100-0036701,BHKW BASF F17,Nordrhein-Westfalen,Erdgas,NaN,2016,2016,1.968,in Betrieb,1,BASF Personal Care and Nutrition GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.968


In [142]:
plants_final2.sort_values('activepower', ascending=False)

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
230,SN70015796,Boxberg Block N,Sachsen,Braunkohle,NaN,2012,1979,2470.0,in Betrieb,4,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2470.0
322,NW300-0326774,Niederaußem,Nordrhein-Westfalen,Braunkohle,Ja,2003,1963,3359.0,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2220.0
275,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,Ja,2012,1972,4211.0,in Betrieb,7,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2120.0
344,NW300-0877384,Weisweiler,Nordrhein-Westfalen,Braunkohle,Ja,2006,1955,2619.0,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2040.0
217,BB45025564,Kraftwerk Jänschwalde Block A,Brandenburg,Braunkohle,Ja,1989,1981,3000.0,in Betrieb,6,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327,NW900-0884101,KW Lünen,Nordrhein-Westfalen,Steinkohle,Ja,1970,1962,473.0,stillgelegt,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
335,BWpf-450-1479296-00000000,Bestandsanlage_HKW_Aalen,Baden-Württemberg,Erdgas,Ja,1960,1960,15.0,stillgelegt,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
339,NW900-0271161,Shamrock,Nordrhein-Westfalen,Steinkohle,Ja,1957,1957,132.0,stillgelegt,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
341,NW100-0081105,Frimmersdorf,Nordrhein-Westfalen,Braunkohle,Nein,1970,1957,2008.0,stillgelegt,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [143]:
plants_final2[pd.isnull(plants_final2['plantid'])]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower


In [144]:
stammdaten

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,gerpower,fullsupply,grosspower
0,NaN,SEE915851127786,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
1,06-08-80920445,SEE946736241334,73432,Aalen,Palm Allee,1,Baden-Württemberg,2021.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
2,NaN,SEE930596800480,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
3,NaN,SEE929797382345,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
4,NaN,SEE988046628214,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1991.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1533,03-01-01010157480,SEE975094128629,38114.0,Braunschweig,Reiherstraße,3,Niedersachsen,NaN,"Erdgas, Erdölgas",NaN,Volleinspeisung,22.400
1534,NaN,SEE926418118239,49479.0,Ibbenbüren,Groner Allee,76,Nordrhein-Westfalen,NaN,"Erdgas, Erdölgas",NaN,Teileinspeisung (einschließlich Eigenverbrauch),1.853
1535,06-08-2142201,SEE937157344278,74399.0,Walheim,Mühlstraße,1,Baden-Württemberg,NaN,Steinkohlen,NaN,Volleinspeisung,107.000
1536,06-08-4124448,SEE981220191160,77704.0,Oberkirch,Hauptstraße,2,Baden-Württemberg,NaN,Steinkohlen,NaN,Teileinspeisung (einschließlich Eigenverbrauch),20.000


In [145]:
#plants_final2.sort_values("activepower", ascending=False)

In [146]:
#stammdaten2.dtypes

In [147]:
#stammdaten3.sort_values(by=['plantid'], ascending=True)

In [148]:
#stammdaten3 = stammdaten2.drop(DROP_ST, axis=1)

In [149]:
#stammdaten4 = stammdaten3.drop_duplicates(['bnaid'])

In [150]:
#stammdaten4

In [151]:
#list(stammdaten2)

In [152]:
'''


"CREATE TABLE blocks(plantid TEXT, blockid TEXT NOT NULL PRIMARY KEY, blockdescription TEXT, federalstate TEXT, energysource TEXT, initialop INTEGER, chp TEXT,
blockname TEXT, netpower REAL, state TEXT, endop TEXT, company TEXT, FOREIGN KEY (blockid) REFERENCES addresses(blockid) ON DELETE CASCADE);"
662-01|BNA0164|Vattenfall GmbH|Brunsbüttel|Schleswig-Holstein|GT D|0.0|stillgelegt|Mineralölprodukte|Nein|63.5|201

'''


'\n\n\n"CREATE TABLE blocks(plantid TEXT, blockid TEXT NOT NULL PRIMARY KEY, blockdescription TEXT, federalstate TEXT, energysource TEXT, initialop INTEGER, chp TEXT,\nblockname TEXT, netpower REAL, state TEXT, endop TEXT, company TEXT, FOREIGN KEY (blockid) REFERENCES addresses(blockid) ON DELETE CASCADE);"\n662-01|BNA0164|Vattenfall GmbH|Brunsbüttel|Schleswig-Holstein|GT D|0.0|stillgelegt|Mineralölprodukte|Nein|63.5|201\n\n'

In [153]:
#|664-02|IKW

In [305]:
blocks2 = blocks[['blockid', 'plantid', 'plantname', 'federalstate', 'energysource', 'initialop', 'chp', 'power', 'state', 'endop', 'company']]

In [306]:
blocks2['initialop'] = blocks2['initialop'].astype('Int32')
blocks2['endop'] = blocks2['endop'].astype('Int32')

/tmp/ipykernel_1714311/1874890394.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2['initialop'] = blocks2['initialop'].astype('Int32')
/tmp/ipykernel_1714311/1874890394.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2['endop'] = blocks2['endop'].astype('Int32')


In [307]:
blocks2.loc[blocks2.blockid == "06-08-2948214"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company


In [308]:
blocks2.loc[blocks2.blockid == "SEE982928435068"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
60,SEE982928435068,BE172656,GuD Marzahn Dampfturbine,Berlin,Erdgas,2020,NaN,69.3,in Betrieb,<NA>,BEW Berliner Energie und Wärme AG


In [309]:
blocks2['chp'] = blocks2['chp'].fillna('Nein')

/tmp/ipykernel_1714311/2832682086.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2['chp'] = blocks2['chp'].fillna('Nein')


In [310]:
list(blocks2)

['blockid',
 'plantid',
 'plantname',
 'federalstate',
 'energysource',
 'initialop',
 'chp',
 'power',
 'state',
 'endop',
 'company']

In [311]:
stammdaten['plz'] = stammdaten['plz'].astype('Int32')

In [312]:
stammdaten.loc[stammdaten.duplicated(subset=['blockid'])]

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,gerpower,fullsupply,grosspower


In [313]:
stammdaten.drop_duplicates(subset="blockid", inplace=True)
blocks2.drop_duplicates(subset="blockid", inplace=True)
plants_final2.drop_duplicates(subset="plantid", inplace=True)

/tmp/ipykernel_1714311/3735086467.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2.drop_duplicates(subset="blockid", inplace=True)


In [314]:
stammdaten2.loc[stammdaten.blockid == "SEE915851127786"]

,blockid,plz,place,street,streetnum,federalstate
0,SEE915851127786,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen


In [315]:
stammdaten2 = stammdaten[['blockid', 'plz', 'place', 'street', 'streetnum', 'federalstate']]

In [316]:
#blocks2.loc['C', 'x'] = "BNA1949"

In [317]:
stammdaten2.to_csv("stammdaten_nh_new.csv", index=False, header=False)
blocks2.to_csv("blocks_nh_2.csv", index=False, header=False)
blocks2.to_csv("blocks_new_nh.csv", index=False, header=False)
plants_final2.to_csv("plants_nh_2.csv", index=False, header=False)
stammdaten2.to_csv("stammdaten.csv", index=False)
blocks2.to_csv("blocks_2.csv", index=False)
plants_final2.to_csv("plants_2.csv", index=False)

In [318]:
#sqlite3 plantwatch.db  "CREATE TABLE plants(plantid TEXT NOT NULL PRIMARY KEY, plantname TEXT, federalstate TEXT, energysource TEXT, chp TEXT, latestexpanded INT, initialop INT, totalpower REAL, state TEXT, blockcount INT,  company TEXT, plz TEXT, place TEXT, street TEXT, number TEXT, latitude REAL, longitude REAL, activepower REAL, energy_2015 INTEGER,  energy_2016 INTEGER,  energy_2017 INTEGER,  energy_2018 INTEGER,  energy_2019 INTEGER, energy_2020 INTEGER, energy_2021 INTEGER, co2_2007 INTEGER,  co2_2008 INTEGER,  co2_2009 INTEGER,  co2_2010 INTEGER,  co2_2011 INTEGER,  co2_2012 INTEGER,  co2_2013 INTEGER,  co2_2014 INTEGER,  co2_2015 INTEGER,  co2_2016 INTEGER,  co2_2017 INTEGER,  co2_2018 INTEGER, co2_2019 INTEGER, co2_2020 INTEGER, FOREIGN KEY (plantid) REFERENCES blocks(plantid) ON DELETE CASCADE);"


In [161]:
blocks2.dropna(subset=['plantid'])

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1,SEE946736241334,BWpf-450-80920445-00000000,HKW Aalen,Baden-Württemberg,Erdgas,2021.0,NaN,78.302,in Betrieb,NaN,Palm Power GmbH & Co. KG
6,SEE902796244388,SD661-59,Turbine 4,Niedersachsen,Erdgas,1970.0,NaN,11.000,in Betrieb,NaN,Sappi Alfeld
8,SEE944503176377,BWpf-450-1741292-00000000,Heizkraftwerk Altbach/Deizisau HKW 1,Baden-Württemberg,Steinkohle,1985.0,NaN,433.000,Netzreserve,NaN,EnBW AG
9,SEE971282697380,BWpf-450-1741292-00000000,Heizkraftwerk Altbach/Deizisau Block 4 GT A (s...,Baden-Württemberg,Erdgas,1971.0,NaN,45.000,in Betrieb,NaN,EnBW AG
10,SEE963975943188,BWpf-450-1741292-00000000,Heizkraftwerk Altbach/Deizisau GT B,Baden-Württemberg,Erdgas,1973.0,NaN,57.000,in Betrieb,NaN,EnBW AG
...,...,...,...,...,...,...,...,...,...,...,...
1532,SEE996990805407,NI01010157480,HKW Mitte Kessel 1,Niedersachsen,Steinkohle,1985.0,Ja,44.500,stillgelegt,2024.0,NaN
1533,SEE975094128629,NI01010157480,HKW Mitte Kessel 12,Niedersachsen,Erdgas,1971.0,Ja,20.000,stillgelegt,2024.0,NaN
1535,SEE937157344278,BWpf-450-2142201-00000000,Kraftwerk Walheim Block 1,Baden-Württemberg,Steinkohle,1964.0,Nein,96.000,stillgelegt,2024.0,NaN
1536,SEE981220191160,BWpf-450-4124448-00000000,HKW Oberkirch,Baden-Württemberg,Steinkohle,1987.0,Ja,18.500,stillgelegt,2024.0,NaN


In [162]:
blocks2.shape

(1538, 11)

In [163]:
blocks2.dropna(subset=['plantid']).shape

(1096, 11)

In [164]:
#stammdaten.dropna(subset=['sseid', 'bnaid']).sort_values(by=['sseid'], ascending=False)

In [165]:
stammdaten

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,gerpower,fullsupply,grosspower
0,NaN,SEE915851127786,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
1,06-08-80920445,SEE946736241334,73432,Aalen,Palm Allee,1,Baden-Württemberg,2021.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
2,NaN,SEE930596800480,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
3,NaN,SEE929797382345,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
4,NaN,SEE988046628214,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1991.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1533,03-01-01010157480,SEE975094128629,38114.0,Braunschweig,Reiherstraße,3,Niedersachsen,NaN,"Erdgas, Erdölgas",NaN,Volleinspeisung,22.400
1534,NaN,SEE926418118239,49479.0,Ibbenbüren,Groner Allee,76,Nordrhein-Westfalen,NaN,"Erdgas, Erdölgas",NaN,Teileinspeisung (einschließlich Eigenverbrauch),1.853
1535,06-08-2142201,SEE937157344278,74399.0,Walheim,Mühlstraße,1,Baden-Württemberg,NaN,Steinkohlen,NaN,Volleinspeisung,107.000
1536,06-08-4124448,SEE981220191160,77704.0,Oberkirch,Hauptstraße,2,Baden-Württemberg,NaN,Steinkohlen,NaN,Teileinspeisung (einschließlich Eigenverbrauch),20.000


In [166]:
blocks.loc[blocks["blockid"] == "BNA0645"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower


In [167]:
#bm2.to_csv("bpm.csv", index=False)

In [168]:
#blocks

In [169]:
#blocks.groupby('Energieträger').count()

In [170]:
#pl4 = blocks.loc[blocks['Energieträger'] == "Mineralölprodukte"]

In [171]:
#pl4

In [172]:
#pd.set_option('display.max_rows', None)

In [173]:
#pl4.loc[pl4['Energieträger'] == "Mineralölprodukte"].sort_values(["Bundesland", "Unternehmen", "Kraftwerksname"])

In [174]:
#plants_final2

In [175]:
blocks2.sort_values('power', ascending=False)[0:20]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1482,SEE943690268513,SD666-16,Isar 2,Bayern,Kernenergie,1988.0,Nein,1410.0,stillgelegt,2023.0,NaN
1423,SEE951462745445,SD666-14,Brokdorf,Schleswig-Holstein,Kernenergie,1986.0,Nein,1410.0,stillgelegt,2021.0,NaN
1401,BNA0802,SD666-12,Kernkraftwerk Philippsburg 2,Baden-Württemberg,Kernenergie,1985.0,Nein,1402.0,stillgelegt,2019.0,NaN
1426,SEE930752846949,SD666-13,Grohnde,Niedersachsen,Kernenergie,1984.0,Nein,1360.0,stillgelegt,2021.0,NaN
1486,SEE944567587799,SD666-17,Emsland A,Niedersachsen,Kernenergie,1988.0,Nein,1336.0,stillgelegt,2023.0,NaN
1491,SEE985577062814,SD666-18,GKN II,Baden-Württemberg,Kernenergie,1989.0,Nein,1310.0,stillgelegt,2023.0,NaN
1432,SEE927528071629,SD666-11,Gundremmingen C,Bayern,Kernenergie,1984.0,Nein,1288.0,stillgelegt,2021.0,NaN
1374,BNA0381,SD666-11,Kernkraft Gundremmingen,Bayern,Kernenergie,1984.0,Nein,1284.0,stillgelegt,2017.0,NaN
1336,BNA0355,SD666-15,Grafenrheinfeld,Bayern,Kernenergie,1982.0,Nein,1275.0,stillgelegt,2015.0,NaN
393,SEE993592183001,NW100-0248923,Neurath G,Nordrhein-Westfalen,Braunkohle,2012.0,NaN,1060.0,in Betrieb,NaN,RWE AG


In [293]:
blocks2.sort_values('power', ascending=False)[0:20]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1482,SEE943690268513,SD666-16,Isar 2,Bayern,Kernenergie,1988,Nein,1410.0,stillgelegt,2023,NaN
1423,SEE951462745445,SD666-14,Brokdorf,Schleswig-Holstein,Kernenergie,1986,Nein,1410.0,stillgelegt,2021,NaN
1401,BNA0802,SD666-12,Kernkraftwerk Philippsburg 2,Baden-Württemberg,Kernenergie,1985,Nein,1402.0,stillgelegt,2019,NaN
1426,SEE930752846949,SD666-13,Grohnde,Niedersachsen,Kernenergie,1984,Nein,1360.0,stillgelegt,2021,NaN
1486,SEE944567587799,SD666-17,Emsland A,Niedersachsen,Kernenergie,1988,Nein,1336.0,stillgelegt,2023,NaN
1491,SEE985577062814,SD666-18,GKN II,Baden-Württemberg,Kernenergie,1989,Nein,1310.0,stillgelegt,2023,NaN
1432,SEE927528071629,SD666-11,Gundremmingen C,Bayern,Kernenergie,1984,Nein,1288.0,stillgelegt,2021,NaN
1374,BNA0381,SD666-11,Kernkraft Gundremmingen,Bayern,Kernenergie,1984,Nein,1284.0,stillgelegt,2017,NaN
1336,BNA0355,SD666-15,Grafenrheinfeld,Bayern,Kernenergie,1982,Nein,1275.0,stillgelegt,2015,NaN
393,SEE993592183001,NW100-0248923,Neurath G,Nordrhein-Westfalen,Braunkohle,2012,NaN,1060.0,in Betrieb,<NA>,RWE AG
